# Routing Queries Across Multiple PDF Files with Metadata

**Files:**
- `pharma-blob-sample.pdf`: Cytiva bundle, 10 pages
- `pharma-blob-test.pdf`: West Pharmaceutical Services bundle, 10 pages, from a different supplier with different document names

**Model:** Mistral-7B-Instruct v0.2 (local, llama.cpp). No embeddings are used for routing.
**Author:** Bashar

The router decides **which pages, in which files, could answer a question before any search is run**:

1. Every page is stored with metadata (`file_id`, `user_id`, `doc_type`, `year`, `filename`, `page_number`, `text`). `doc_type` comes from `classify_doc_type_llm()`.
2. `classify_query_llm()` predicts which document type the question is about.
3. Pages are filtered to that `doc_type`, then ranked by keyword overlap with the question so the best page comes first.
4. If the query type is `unknown` or nothing matches, a keyword search over all pages is the fallback.

Ten test questions, each with a known answer location, measure how often the route contains the right page.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

## Step 0: Install dependencies

In [ ]:
%pip install -q llama-cpp-python==0.3.19 --only-binary=llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
%pip install -q pymupdf huggingface_hub

## Step 1: Model and the two PDFs

In [1]:
import os, json, re, time, uuid
from functools import lru_cache
import llama_cpp
from llama_cpp import Llama

MODEL_PATH = os.environ.get("MISTRAL_GGUF", "/content/models/mistral-7b-instruct-v0.2.Q4_K_M.gguf")
if not os.path.exists(MODEL_PATH):
    from huggingface_hub import hf_hub_download
    hf_hub_download("TheBloke/Mistral-7B-Instruct-v0.2-GGUF", "mistral-7b-instruct-v0.2.Q4_K_M.gguf",
                    local_dir=os.path.dirname(MODEL_PATH))

PDF_DIR = os.environ.get("PDF_DIR", "/content")
DRIVE_IDS = {"pharma-blob-sample.pdf": "1fJ7DszcjB_TerDTAhXfAw74VqJBElTyw",   # course sample files, shared publicly
             "pharma-blob-test.pdf": "1atOj_CzvPRm-CskbhvJdzlJIpcEanVu1"}
for name, drive_id in DRIVE_IDS.items():
    path = os.path.join(PDF_DIR, name)
    if not os.path.exists(path):
        import urllib.request
        urllib.request.urlretrieve(f"https://drive.google.com/uc?export=download&id={drive_id}", path)

N_GPU_LAYERS = -1 if llama_cpp.llama_supports_gpu_offload() else 0
llm = Llama(model_path=MODEL_PATH, n_ctx=4096, n_gpu_layers=N_GPU_LAYERS, seed=42, verbose=False)
print(f"Mistral loaded on {'GPU' if N_GPU_LAYERS else 'CPU'}")

Mistral loaded on CPU


## Step 2: Capture page-level metadata at upload

The course's structure, one record per page. Two small changes:

- **`file_id` is derived from the filename** (`uuid5`) instead of being random (`uuid4`), so it stays the same every time the notebook runs and the output can be compared between runs.
- **`year` is set per file**, from the newest issue date in each file, instead of one hard-coded year for everything.

In [2]:
import fitz  # PyMuPDF

FILE_YEARS = {"pharma-blob-sample.pdf": "2024",   # newest issue date: 15 January 2024
              "pharma-blob-test.pdf": "2025"}     # newest issue date: 12 February 2025
user_id = "xyz"

pdf_metadata_store = []
for filename, year in FILE_YEARS.items():
    file_id = str(uuid.uuid5(uuid.NAMESPACE_URL, filename))
    with fitz.open(os.path.join(PDF_DIR, filename)) as pdf:
        for i, page in enumerate(pdf):
            pdf_metadata_store.append({
                "file_id": file_id, "user_id": user_id, "doc_type": "unknown", "year": year,
                "filename": filename, "page_number": i + 1, "text": page.get_text(),
            })

print(f"{len(pdf_metadata_store)} pages stored")
print({k: v for k, v in pdf_metadata_store[10].items() if k != "text"})

20 pages stored
{'file_id': '06de3fe2-dd6b-5a9a-b0d0-57ca0b1f2987', 'user_id': 'xyz', 'doc_type': 'unknown', 'year': '2025', 'filename': 'pharma-blob-test.pdf', 'page_number': 1}


## Step 3: The two LLM classifiers

Both use the same eight labels from the task. llama.cpp restricts the answer to a JSON object whose `doc_type` must be one of them, so the router can never receive an unexpected label.

The type definitions are the ones written for the Cytiva file in the previous tasks, with "or email" added to `cover_letter` and "or batch" / "composition" added to the certificate and material types. They were **not** tuned to the West file. Its document names ("Animal Origin Statement", "Vendor Qualification Record", "Compliance Bulletin") never appear in them, which makes the West file a fair test of whether the labels generalise.

In [3]:
DOC_TYPES = ["cover_letter", "certificate_of_quality", "packaging_specification", "bse_tse_declaration",
             "material_description", "supplier_qualification", "chain_of_custody", "unknown"]

TYPE_DEFINITIONS = '''- cover_letter: a letter or email addressed to the reader (greeting, sign-off), e.g. about storage, shipping or what a package contains
- certificate_of_quality: certifies one product lot or batch (lot/batch numbers, manufacture/expiry dates, release tests)
- packaging_specification: packaging materials, configuration, packaging part numbers, drawings
- bse_tse_declaration: statement that products contain no animal-origin materials (BSE/TSE)
- material_description: materials of construction, composition and physical properties of a product
- supplier_qualification: supplier approval status, audits, certifications, approved products
- chain_of_custody: where assemblies are manufactured and how they move to the customer
- unknown: none of the above'''

SCHEMA = {"type": "object", "properties": {"doc_type": {"type": "string", "enum": DOC_TYPES}}, "required": ["doc_type"]}

@lru_cache(maxsize=None)
def ask_doc_type(prompt):
    response = llm.create_chat_completion(messages=[{"role": "user", "content": prompt}],
                                          response_format={"type": "json_object", "schema": SCHEMA},
                                          temperature=0.0, max_tokens=30)
    return json.loads(response["choices"][0]["message"]["content"])["doc_type"]

def classify_doc_type_llm(page_text):
    return ask_doc_type(f'''Classify this page from a bundle of pharmaceutical supplier documents.

Document types:
{TYPE_DEFINITIONS}

PAGE:
{page_text[:3000]}

Return JSON with "doc_type".''')

def classify_query_llm(query):
    return ask_doc_type(f'''A user is asking a question about a set of pharmaceutical supplier documents. Which type of document is most likely to contain the answer?

Document types:
{TYPE_DEFINITIONS}

QUESTION: {query}

Return JSON with "doc_type". Use "unknown" if the question does not point to one type.''')

## Step 4: Assign a `doc_type` to every page

Checked against labels I assigned by reading each page. Two West pages do not fit the eight types cleanly, so either of two answers is accepted for them:

- **Page 7 (Compliance Bulletin):** composition and physical properties, so `material_description`.
- **Page 10 (STERIS sterilisation certificate):** none of the eight types, so `unknown`. `certificate_of_quality` is also accepted, since it certifies per-batch results.

In [4]:
import pandas as pd

t0 = time.perf_counter()
for page in pdf_metadata_store:
    page["doc_type"] = classify_doc_type_llm(page["text"])
print(f"{len(pdf_metadata_store)} pages classified in {time.perf_counter() - t0:.0f}s\n")

EXPECTED_PAGE_TYPES = {
    "pharma-blob-sample.pdf": [{"cover_letter"}, {"certificate_of_quality"}, {"certificate_of_quality"},
        {"packaging_specification"}, {"packaging_specification"}, {"bse_tse_declaration"},
        {"material_description"}, {"supplier_qualification"}, {"supplier_qualification"}, {"chain_of_custody"}],
    "pharma-blob-test.pdf": [{"cover_letter"}, {"certificate_of_quality"}, {"certificate_of_quality"},
        {"certificate_of_quality"}, {"packaging_specification"}, {"bse_tse_declaration"},
        {"material_description"}, {"supplier_qualification"}, {"supplier_qualification"},
        {"unknown", "certificate_of_quality"}],
}

page_table = pd.DataFrame([{
    "filename": p["filename"], "page": p["page_number"],
    "title": next((l.strip() for l in p["text"].splitlines() if l.strip() and not l.startswith("West Pharma")), "")[:45],
    "predicted doc_type": p["doc_type"],
    "correct": "yes" if p["doc_type"] in EXPECTED_PAGE_TYPES[p["filename"]][p["page_number"] - 1] else "NO",
} for p in pdf_metadata_store])
print("Pages classified correctly:", (page_table["correct"] == "yes").sum(), "/", len(page_table))
page_table

20 pages classified in 273s

Pages classified correctly: 19 / 20


,filename,page,title,predicted doc_type,correct
0,pharma-blob-sample.pdf,1,Cytiva,cover_letter,yes
1,pharma-blob-sample.pdf,2,Certificate of Quality,certificate_of_quality,yes
2,pharma-blob-sample.pdf,3,Certificate of Quality,certificate_of_quality,yes
3,pharma-blob-sample.pdf,4,Packaging Component Specification,packaging_specification,yes
4,pharma-blob-sample.pdf,5,Packaging Component Specification (continued),packaging_specification,yes
5,pharma-blob-sample.pdf,6,Declaration Regarding Transmissible Spongifor,bse_tse_declaration,yes
6,pharma-blob-sample.pdf,7,Material Description Sheet,material_description,yes
7,pharma-blob-sample.pdf,8,Supplier Qualification Record,supplier_qualification,yes
8,pharma-blob-sample.pdf,9,Supplier Qualification Record (continued),supplier_qualification,yes
9,pharma-blob-sample.pdf,10,Cytiva,chain_of_custody,yes


## Step 5: The router

- **Filter:** keep pages whose `doc_type` equals the predicted type.
- **Rank:** score each kept page by how many of the question's words (and any specific numbers in it, such as batch numbers) appear on the page, so the best page is listed first.
- **Fallback:** if the predicted type is `unknown`, or no page has that type, rank *all* pages by keyword overlap instead, using the question's words plus the task's domain keywords, and return the best few.

In [5]:
STOPWORDS = set('''a an and are as at be by can could did do does for from how in is it its list listed me of on or
show tell that the their them there these this those to was were what when where which who whose why will with you
your document documents'''.split())
DOMAIN_KEYWORDS = ["part number", "article number", "specification", "certificate", "lot", "sterilization",
                   "bse", "supplier"]

def query_terms(query):
    words = [w for w in re.findall(r"[a-z0-9/<>.-]+", query.lower()) if w not in STOPWORDS and len(w) > 2]
    return [w.strip(".") for w in words]

def keyword_score(query, text):
    text_l = text.lower()
    score = sum(text_l.count(t) for t in query_terms(query))
    # exact identifiers in the question (batch, lot, document numbers) are strong evidence
    score += 10 * sum(1 for t in re.findall(r"\b[A-Z0-9][A-Z0-9-]{5,}\b", query) if t in text)
    return score

def public(page, score):
    return {**page, "keyword_score": score}

def route(query, top_n_fallback=3):
    predicted = classify_query_llm(query)
    matched = [p for p in pdf_metadata_store if p["doc_type"] == predicted] if predicted != "unknown" else []
    used_fallback = not matched
    if used_fallback:
        pool = pdf_metadata_store
        scored = sorted(((keyword_score(query + " " + " ".join(DOMAIN_KEYWORDS), p["text"]), p) for p in pool),
                        key=lambda x: -x[0])[:top_n_fallback]
    else:
        scored = sorted(((keyword_score(query, p["text"]), p) for p in matched), key=lambda x: -x[0])
    return {"query": query, "predicted_doc_type": predicted, "used_fallback": used_fallback,
            "matched_documents": [public(p, s) for s, p in scored]}

## Step 6: Test questions

Each question has a known answer location (file, page). The router is scored on:

- **Answer page routed:** whether any matched page is an answer page.
- **Answer page ranked first:** whether the best match is an answer page.
- **Pages to search:** how many pages are passed on out of 20.

Q1 is the task's example. It is deliberately ambiguous, because part numbers appear in five different document types, so it is not scored.

In [6]:
TESTS = [
    ("What are the part numbers listed in this document?", None),
    ("What are the BSE/TSE compliance dates?", [("pharma-blob-sample.pdf", 6), ("pharma-blob-test.pdf", 6)]),
    ("What gamma irradiation dose did batch D000291847 receive?", [("pharma-blob-test.pdf", 10)]),
    ("Which sub-suppliers does the vendor use?", [("pharma-blob-sample.pdf", 9), ("pharma-blob-test.pdf", 9)]),
    ("What materials is the stopper made of?", [("pharma-blob-test.pdf", 7)]),
    ("When was the supplier last audited?", [("pharma-blob-sample.pdf", 8), ("pharma-blob-test.pdf", 8)]),
    ("What is the expiry date of batch D000291852?", [("pharma-blob-test.pdf", 3)]),
    ("How should the flow kits be stored?", [("pharma-blob-sample.pdf", 1)]),
    ("Which documents are included in West's SDF package?", [("pharma-blob-test.pdf", 1)]),
    ("Show me all supplier documentation", None),
]

routes, rows = [], []
for query, answer_pages in TESTS:
    r = route(query)
    routes.append(r)
    found = [(d["filename"], d["page_number"]) for d in r["matched_documents"]]
    rows.append({
        "Question": query,
        "Predicted doc_type": r["predicted_doc_type"] + (" (fallback)" if r["used_fallback"] else ""),
        "Pages to search": len(found),
        "Answer page routed": "n/a" if answer_pages is None else ("yes" if set(found) & set(answer_pages) else "NO"),
        "Answer page ranked first": "n/a" if answer_pages is None else ("yes" if found and found[0] in answer_pages else "NO"),
        "Best match": f"{found[0][0].replace('pharma-blob-', '')} p{found[0][1]}" if found else "none",
    })
results_table = pd.DataFrame(rows)
scored = results_table[results_table["Answer page routed"] != "n/a"]
print(f"Answer page routed: {(scored['Answer page routed'] == 'yes').sum()}/{len(scored)}   "
      f"ranked first: {(scored['Answer page ranked first'] == 'yes').sum()}/{len(scored)}   "
      f"average pages to search: {results_table['Pages to search'].mean():.1f} of {len(pdf_metadata_store)}")
results_table

Answer page routed: 7/8   ranked first: 5/8   average pages to search: 3.8 of 20


,Question,Predicted doc_type,Pages to search,Answer page routed,Answer page ranked first,Best match
0,What are the part numbers listed in this docum...,packaging_specification,3,n/a,n/a,sample.pdf p4
1,What are the BSE/TSE compliance dates?,bse_tse_declaration,2,yes,yes,sample.pdf p6
2,What gamma irradiation dose did batch D0002918...,certificate_of_quality,6,yes,yes,test.pdf p10
3,Which sub-suppliers does the vendor use?,supplier_qualification,5,yes,NO,test.pdf p8
4,What materials is the stopper made of?,material_description,2,yes,yes,test.pdf p7
5,When was the supplier last audited?,supplier_qualification,5,yes,NO,sample.pdf p9
6,What is the expiry date of batch D000291852?,certificate_of_quality,6,yes,yes,test.pdf p3
7,How should the flow kits be stored?,cover_letter,1,yes,yes,sample.pdf p1
8,Which documents are included in West's SDF pac...,packaging_specification,3,NO,NO,test.pdf p5
9,Show me all supplier documentation,supplier_qualification,5,n/a,n/a,sample.pdf p8


## Step 7: Final output in the task's format

The full routing result for every question, with each matched page's complete metadata, is saved to `routing_results.json`. Below is the example from the task.

In [7]:
with open("routing_results.json", "w") as f:
    json.dump([{k: v for k, v in r.items() if k != "used_fallback"} for r in routes], f, indent=2)

example = routes[0]
print(json.dumps({
    "query": example["query"],
    "predicted_doc_type": example["predicted_doc_type"],
    "matched_documents": [{**{k: d[k] for k in ["file_id", "user_id", "doc_type", "year", "filename", "page_number"]},
                           "text": d["text"][:120] + "..."} for d in example["matched_documents"]],
}, indent=2))

{
  "query": "What are the part numbers listed in this document?",
  "predicted_doc_type": "packaging_specification",
  "matched_documents": [
    {
      "file_id": "1c89f2db-a339-5d5e-92b1-dc0729404556",
      "user_id": "xyz",
      "doc_type": "packaging_specification",
      "year": "2024",
      "filename": "pharma-blob-sample.pdf",
      "page_number": 4,
      "text": "Packaging Component Specification\nDocument Number:\nPKG-SPEC-2023-0847\nRevision:\nC\nEffective Date:\n2023-11-08\nComponent:\n..."
    },
    {
      "file_id": "1c89f2db-a339-5d5e-92b1-dc0729404556",
      "user_id": "xyz",
      "doc_type": "packaging_specification",
      "year": "2024",
      "filename": "pharma-blob-sample.pdf",
      "page_number": 5,
      "text": "Packaging Component Specification (continued)\nDocument Number: PKG-SPEC-2023-0847, Page 2 of 2\n4. Configuration Change H..."
    },
    {
      "file_id": "06de3fe2-dd6b-5a9a-b0d0-57ca0b1f2987",
      "user_id": "xyz",
      "doc_type": "pac

## Findings

Run with Mistral-7B on a laptop CPU: 20 page classifications plus 10 query classifications, about 5 minutes.

| Measure | Result |
|---|---|
| Pages given the right `doc_type` | 19 of 20 |
| Questions where an answer page was routed | 7 of 8 scored |
| Questions where an answer page was ranked first | 5 of 8 |
| Pages to search per question | 3.8 of 20 on average |

**1. Page metadata did most of the work, and it generalised to a new supplier.** The type definitions were written for the Cytiva file, yet the West file's differently named documents were labelled correctly: "Animal Origin Statement" as `bse_tse_declaration`, "Vendor Qualification Record" as `supplier_qualification`, and "Compliance Bulletin" as `material_description`. This mattered for routing. The West Animal Origin Statement never uses the words "BSE", "TSE" or "compliance" (its keyword score for the BSE/TSE question was 0), so a keyword search would have missed it, but the `doc_type` filter found it. Routing cut the search from 20 pages to about 4.

**2. The query classifier is the weak link.** "Which documents are included in West's SDF package?" was routed to `packaging_specification`, because the model read "package" as packaging. Here "SDF package" means the bundle of documents described in the cover email. The answer page was never searched.

**3. A wrong page label compounds a wrong query label.** The one misclassified page is that same West cover email, labelled `supplier_qualification` because it lists the package contents and talks about compliance. So even a correctly classified query (`cover_letter`) would have missed it. The mislabelled page also shows up as an extra match for every supplier question.

**4. Keyword ranking inside the routed set is crude.** For the sub-supplier and last-audit questions, the right page was routed but ranked second or third. Counting word matches rewards pages that say "vendor" or "supplier" often, not the page that answers the question. Specific identifiers in the question work well: batch numbers put the STERIS certificate and the batch table first.

**5. Vague and ambiguous questions route somewhere, but not usefully.** "What are the part numbers listed in this document?" (the task's example) went to `packaging_specification`. Part numbers also appear in the cover letter, material description and supplier records, so there is no single right type. "Show me all supplier documentation" went to `supplier_qualification`, although every document in both bundles is supplier documentation.

**How to improve it**
- Let the query classifier return its top two types, or fall back to a wider search when the routed pages share no words with the question.
- Group pages into documents (from the segmentation step) and return whole documents, so a two-page record is never half-retrieved.
- Replace keyword ranking with embedding search *within* the routed pages.
- Store more metadata at upload (supplier, product or item number, batch numbers) so questions that name them can be filtered exactly.